In [2]:
import pandas as pd

In [17]:
# ---------- 1) Download Census ACS 5-year county data (2023) ----------
poverty_url = "https://api.census.gov/data/2023/acs/acs5/subject?get=NAME,S1701_C03_001E&for=county:*" # Poverty status in the past 12 months below poverty level
income_url  = "https://api.census.gov/data/2023/acs/acs5/subject?get=NAME,S1901_C01_012E&for=county:*" # Median household income in the past 12 months (in 2023 inflation-adjusted dollars)
edu_url     = "https://api.census.gov/data/2023/acs/acs5/subject?get=NAME,S1501_C02_015E&for=county:*" # Educational attainment for the population 25 years and over: Bachelor's degree or higher

poverty = pd.read_json(poverty_url) 
income  = pd.read_json(income_url)
edu     = pd.read_json(edu_url)

print(poverty.head(5))

                         0               1      2       3
0                     NAME  S1701_C03_001E  state  county
1  Autauga County, Alabama            10.7     01     001
2  Baldwin County, Alabama            10.5     01     003
3  Barbour County, Alabama            21.9     01     005
4     Bibb County, Alabama            20.5     01     007


In [18]:
# ---------- 2) Fix headers ----------
poverty.columns = poverty.iloc[0] # Set the first row as column headers
poverty = poverty[1:] # Remove the first row which is now the header
print(poverty.head(5))

#do the same for income and edu
income.columns = income.iloc[0]
income = income[1:]

edu.columns = edu.iloc[0]
edu = edu[1:]

0                     NAME S1701_C03_001E state county
1  Autauga County, Alabama           10.7    01    001
2  Baldwin County, Alabama           10.5    01    003
3  Barbour County, Alabama           21.9    01    005
4     Bibb County, Alabama           20.5    01    007
5   Blount County, Alabama           14.1    01    009


In [20]:
# ---------- 3) Create county FIPS ----------
for df in [poverty, income, edu]:
    df["FIPS"] = df["state"] + df["county"]

print(poverty.head(5)) # Check the new FIPS column
print(income.head(5))
print(edu.head(5))

0                     NAME S1701_C03_001E state county   FIPS
1  Autauga County, Alabama           10.7    01    001  01001
2  Baldwin County, Alabama           10.5    01    003  01003
3  Barbour County, Alabama           21.9    01    005  01005
4     Bibb County, Alabama           20.5    01    007  01007
5   Blount County, Alabama           14.1    01    009  01009
0                     NAME S1901_C01_012E state county   FIPS
1  Autauga County, Alabama          69841    01    001  01001
2  Baldwin County, Alabama          75019    01    003  01003
3  Barbour County, Alabama          44290    01    005  01005
4     Bibb County, Alabama          51215    01    007  01007
5   Blount County, Alabama          61096    01    009  01009
0                     NAME S1501_C02_015E state county   FIPS
1  Autauga County, Alabama           28.3    01    001  01001
2  Baldwin County, Alabama           32.8    01    003  01003
3  Barbour County, Alabama           11.5    01    005  01005
4     Bi

In [21]:
# ---------- 4) Rename variables ----------
poverty = poverty.rename(columns={"S1701_C03_001E": "poverty_rate"})
income  = income.rename(columns={"S1901_C01_012E": "median_income"})
edu     = edu.rename(columns={"S1501_C02_015E": "bachelor_plus_rate"})
print(poverty.head(5))
print(income.head(5))
print(edu.head(5))

0                     NAME poverty_rate state county   FIPS
1  Autauga County, Alabama         10.7    01    001  01001
2  Baldwin County, Alabama         10.5    01    003  01003
3  Barbour County, Alabama         21.9    01    005  01005
4     Bibb County, Alabama         20.5    01    007  01007
5   Blount County, Alabama         14.1    01    009  01009
0                     NAME median_income state county   FIPS
1  Autauga County, Alabama         69841    01    001  01001
2  Baldwin County, Alabama         75019    01    003  01003
3  Barbour County, Alabama         44290    01    005  01005
4     Bibb County, Alabama         51215    01    007  01007
5   Blount County, Alabama         61096    01    009  01009
0                     NAME bachelor_plus_rate state county   FIPS
1  Autauga County, Alabama               28.3    01    001  01001
2  Baldwin County, Alabama               32.8    01    003  01003
3  Barbour County, Alabama               11.5    01    005  01005
4     Bibb

In [ ]:
# ---------- 5) Merge into one dataset ----------
census_df = poverty[["FIPS", "NAME", "poverty_rate"]].merge(
    income[["FIPS", "median_income"]], on="FIPS").merge(
        edu[["FIPS", "bachelor_plus_rate"]], on="FIPS")
print(census_df.head(5))

0   FIPS                     NAME poverty_rate median_income  \
0  01001  Autauga County, Alabama         10.7         69841   
1  01003  Baldwin County, Alabama         10.5         75019   
2  01005  Barbour County, Alabama         21.9         44290   
3  01007     Bibb County, Alabama         20.5         51215   
4  01009   Blount County, Alabama         14.1         61096   

0 bachelor_plus_rate  
0               28.3  
1               32.8  
2               11.5  
3               11.5  
4               15.6  
0
FIPS                  object
NAME                  object
poverty_rate          object
median_income         object
bachelor_plus_rate    object
dtype: object


In [ ]:
# ---------- 6) Convert to numeric ----------
# check data types
print(census_df.dtypes)
census_df["poverty_rate"] = pd.to_numeric(census_df["poverty_rate"], errors="coerce") # Convert poverty_rate to numeric, coercing errors to NaN
census_df["median_income"] = pd.to_numeric(census_df["median_income"], errors="coerce")
census_df["bachelor_plus_rate"] = pd.to_numeric(census_df["bachelor_plus_rate"], errors="coerce")
print(census_df.dtypes) # Check that the columns are now numeric

0
FIPS                  object
NAME                  object
poverty_rate          object
median_income         object
bachelor_plus_rate    object
dtype: object
0
FIPS                   object
NAME                   object
poverty_rate          float64
median_income           int64
bachelor_plus_rate    float64
dtype: object


In [9]:
# ---------- 7) Quick check ----------
print(census_df.shape)
print(census_df.head())

(3222, 5)
0   FIPS                     NAME  poverty_rate  median_income  \
0  01001  Autauga County, Alabama          10.7          69841   
1  01003  Baldwin County, Alabama          10.5          75019   
2  01005  Barbour County, Alabama          21.9          44290   
3  01007     Bibb County, Alabama          20.5          51215   
4  01009   Blount County, Alabama          14.1          61096   

0  bachelor_plus_rate  
0                28.3  
1                32.8  
2                11.5  
3                11.5  
4                15.6  


In [30]:
# ---------- 8) Read NCES CSV files （2022-2023）----------
lea_path = "nces_2022_2023/ccd_lea_029_2223_w_1a_083023.csv"
dropout_path = "nces_2022_2023/CCD_LEA_032_2223_L_1a_PUB_050824.CSV"
grad_path = "nces_2022_2023/CCD_LEA_040_2223_L_1a_PUB_050824.CSV"

lea_df = pd.read_csv(lea_path, low_memory=False) # Read the LEA file, setting low_memory=False to avoid dtype warnings
dropout_df = pd.read_csv(dropout_path, low_memory=False) # Read the dropout file, setting low_memory=False to avoid dtype warnings
grad_df = pd.read_csv(grad_path, low_memory=False) # Read the graduate file, setting low_memory=False to avoid dtype warnings

print("LEA file shape:", lea_df.shape)
print("Dropout file shape:", dropout_df.shape)
print("Graduate file shape:", grad_df.shape)

LEA file shape: (19714, 58)
Dropout file shape: (206074, 15)
Graduate file shape: (344828, 15)


In [13]:
# ---------- 9) Inspect column names----------
print("LEA columns:\n", lea_df.columns.tolist())
print("Dropout columns:\n", dropout_df.columns.tolist())
print("Graduate columns:\n", grad_df.columns.tolist())



LEA columns:
 ['SCHOOL_YEAR', 'FIPST', 'STATENAME', 'ST', 'LEA_NAME', 'STATE_AGENCY_NO', 'UNION', 'ST_LEAID', 'LEAID', 'MSTREET1', 'MSTREET2', 'MSTREET3', 'MCITY', 'MSTATE', 'MZIP', 'MZIP4', 'LSTREET1', 'LSTREET2', 'LSTREET3', 'LCITY', 'LSTATE', 'LZIP', 'LZIP4', 'PHONE', 'WEBSITE', 'SY_STATUS', 'SY_STATUS_TEXT', 'UPDATED_STATUS', 'UPDATED_STATUS_TEXT', 'EFFECTIVE_DATE', 'LEA_TYPE', 'LEA_TYPE_TEXT', 'OUT_OF_STATE_FLAG', 'CHARTER_LEA', 'CHARTER_LEA_TEXT', 'NOGRADES', 'G_PK_OFFERED', 'G_KG_OFFERED', 'G_1_OFFERED', 'G_2_OFFERED', 'G_3_OFFERED', 'G_4_OFFERED', 'G_5_OFFERED', 'G_6_OFFERED', 'G_7_OFFERED', 'G_8_OFFERED', 'G_9_OFFERED', 'G_10_OFFERED', 'G_11_OFFERED', 'G_12_OFFERED', 'G_13_OFFERED', 'G_UG_OFFERED', 'G_AE_OFFERED', 'GSLO', 'GSHI', 'LEVEL', 'IGOFFERED', 'OPERATIONAL_SCHOOLS']
Dropout columns:
 ['SCHOOL_YEAR', 'FIPST', 'STATENAME', 'ST', 'LEA_NAME', 'STATE_AGENCY_NO', 'UNION', 'ST_LEAID', 'LEAID', 'GRADE', 'RACE_ETHNICITY', 'SEX', 'STUDENT_COUNT', 'TOTAL_INDICATOR', 'DMS_FLAG']
G

In [ ]:
# ---------- 10) Merge datasets----------
# Most NCES LEA files share a key like LEAID or LEAID (district ID)
# Find common key columns automatically
common_cols = set(lea_df.columns).intersection(set(dropout_df.columns)).intersection(set(grad_df.columns))
print("\nCommon columns in all 3 files:", common_cols)

# Usually LEAID is the merge key
key = "LEAID" if "LEAID" in lea_df.columns else None # Check if LEAID is in the columns, otherwise set key to None
if key is None:
    raise ValueError("LEAID not found. Need to inspect column names.") # If LEAID is not found, raise an error to inspect column names

# Merge dropout + grad into LEA base
nces_df = lea_df.merge(dropout_df, on=key, how="left", suffixes=("", "_drop")) # Merge dropout data into LEA data, using left join to keep all LEAs, and adding suffixes to distinguish columns
nces_df = nces_df.merge(grad_df, on=key, how="left", suffixes=("", "_grad"))

print("\nMerged NCES shape:", nces_df.shape)
print(nces_df[[key]].head())


Common columns in all 3 files: {'SCHOOL_YEAR', 'FIPST', 'LEAID', 'LEA_NAME', 'STATENAME', 'UNION', 'STATE_AGENCY_NO', 'ST', 'ST_LEAID'}

Merged NCES shape: (5156255, 86)
    LEAID
0  100002
1  100002
2  100002
3  100002
4  100002


In [ ]:
# The merged dataset now contains all the LEA information along with dropout and graduation data, however the data is over 5 million, which is not accurate.
# This indicates the dropout and graduate datasets likely contain multiple rows per district and cause a many to many merge.
